In [ ]:
# General imports:-
import numpy as np;
import pandas as pd;
from scipy.stats import mode, iqr;

import matplotlib.pyplot as plt;
%matplotlib inline
import seaborn as sns;
grid_specs = {'visible':True, 'color': 'lightgrey', 'linewidth': 0.75, 'linestyle': '--'};

from itertools import product, permutations, combinations;
from tqdm.notebook import tqdm;
from warnings import filterwarnings;
filterwarnings('ignore');
from gc import collect;
from termcolor import colored;
from IPython.display import clear_output;

In [ ]:
# Model specific imports:- 
from sklearn_pandas import DataFrameMapper, gen_features;
from sklearn.compose import ColumnTransformer, make_column_selector;
from sklearn.base import TransformerMixin, BaseEstimator;
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer, LabelEncoder, OneHotEncoder;
from sklearn.pipeline import Pipeline;
from sklearn.feature_selection import mutual_info_classif;
from sklearn.metrics import roc_auc_score, r2_score;
from sklearn.linear_model import LogisticRegression, HuberRegressor, LinearRegression;

In [ ]:
# Specific imputation and encoding based import:-
# Taken from the link- https://www.kaggle.com/code/maxsarmento/lb-0-58978-standing-on-the-shoulder-of-giants
!git clone https://github.com/analokmaus/kuma_utils.git;
from sys import path;
path.append("kuma_utils/")
from kuma_utils.preprocessing.imputer import LGBMImputer;

!pip install feature_engine;
from feature_engine.encoding import WoEEncoder, RareLabelEncoder;
clear_output();

## Tabular Playground Series- August 2022 

**This is a binary classification problem for a company's products to predict failure across products from a testing study. 
The analysis measure is the Somer's D/ GINI/ Area under ROC curve.**

This is a continuation from the EDA notebook at the link- https://www.kaggle.com/code/ravi20076/tpsaug22-extensiveeda

We aim to develop features for the model, impute nulls and do feature engineering and develop a model for the assignment.

References- 
* https://www.kaggle.com/code/maxsarmento/lb-0-58978-standing-on-the-shoulder-of-giants
* https://www.kaggle.com/code/ambrosm/tpsaug22-just-guessing

In [ ]:
# Importing the data:-
xytrain = pd.read_csv('../input/tabular-playground-series-aug-2022/train.csv', index_col = 'id');
xtest = pd.read_csv('../input/tabular-playground-series-aug-2022/test.csv', index_col = 'id');
Ftre_Lst = xtest.columns;
Msmt_Ftre_Lst = Ftre_Lst[Ftre_Lst.str.startswith('measurement_')];

# Splitting the dataset into features and target:-
xtrain, ytrain = xytrain[Ftre_Lst], xytrain[['failure']];

print(colored(f"\nTraining and test features-", color= 'blue', attrs= ['bold']));
display(Ftre_Lst);
print(colored(f"\nMeasurement features-", color= 'blue', attrs= ['bold']));
display(Msmt_Ftre_Lst);

print(colored(f"\nTraining and test data sample-\n", color= 'blue', attrs= ['bold']));
display(xtrain.head().style.format(precision=2));
print();
display(xtest.head().style.format(precision=2));

# Step1- Data processing

This is a very important step herewith as evinced throughout the month with this data. We will commence this process using the ideas presented in the link- https://www.kaggle.com/code/maxsarmento/lb-0-58978-standing-on-the-shoulder-of-giants and our observations from the EDA notebook wherein we can construct measurement 17 from measurements 3-9. The same is discussed in several discussion posts as well

We may observe from the EDA notebook that measurements have 2-3 distinct groupings- measurement 17; measurement 16; meaurements 3-9 and measurements 10-15. We may perhaps further investigate these relations here and assess their efficacy in model development processes

We will use the LGBM imputer finally for other null instances above and beyond this measure too. Finally, for loading, we could use product level means/ medians for the loading factor. This factor is almost independent of all other columns and can be considered independently.

**Steps followed:-**
* Imputation of measurement 17 using inear regression from measurements 3-9
* Development of new features using null values and otherwise
* Imputation of remaining nulls in measurements using LGBM Imputer
* Encoding attributes 0 and 1
* Scaling using Robust Scaler
* Assessing univariate characteristics after feature processing

In [ ]:
class Msmt17Imputer(BaseEstimator, TransformerMixin):
    """
    This class aims to do null imputation of measurement 17 column using other measurement columns from measurement3- measurement9
    """;
    
    def __init(self):
        pass
    
    def fit(self, X, y=None, **fit_params): 
        global Msmt_Ftre_Lst;
        self.msmt_cols = Msmt_Ftre_Lst;
        return self;
    
    def transform(self,X, y=None, **transform_params):
        df = X.copy();
        
        # Collating interpolation data with relevant measurement columns:-
        _xy = df[['product_code'] + list(self.msmt_cols[3:10]) + [self.msmt_cols[-1]]];
        _xytrain = _xy.dropna();

        _ = _xy.isna().sum(axis=1);
        _xyimpute = _xy.loc[(_xy.index.isin(_.loc[_ == 1].index)) & (_xy.measurement_17.isna() == True)];

        del _;
        collect();

        # Training the linear model at each product code:-
        products = _xytrain['product_code'].unique();
        model = LinearRegression();

        print(colored(f"\nLinear Regression model",color = 'blue', attrs = ['bold', 'dark']));
        for i, product in tqdm(enumerate(products)):
            print(colored(f"Current product = {product}", color = 'blue', attrs= ['dark', 'bold']));
            model.fit(_xytrain.loc[_xytrain.product_code == product].\
                      drop(['measurement_17', 'product_code'], axis=1), 
                      _xytrain.loc[_xytrain.product_code == product,'measurement_17'].values);

            # Imputing the predictions:-
            preds = model.predict(_xyimpute.loc[_xyimpute.product_code == product].\
                                  drop(['measurement_17', 'product_code'], axis=1));
            r2 = r2_score(_xytrain.loc[_xytrain.product_code == product,'measurement_17'].values, 
                          model.predict(_xytrain.loc[_xytrain.product_code == product].\
                                        drop(['measurement_17', 'product_code'], axis=1)));
            print(colored(f"R-Square for product {product} = {r2:.2%}", color = 'blue'));

            # Updating the imputed values to the relevant rows:-
            _xyimpute.loc[_xyimpute.product_code == product, 'measurement_17'] = preds;
            print(colored(f"{len(preds):,.0f} rows are updated with imputed values"));
            print();
            del preds, r2;
            collect();

        del _xy, _xytrain;
        collect();

        # Updating the dataframe with the null imputation for measurement 17:-
        df = df.merge(_xyimpute[['measurement_17']], how='left', left_index= True, right_index= True, suffixes= ('','_'));
        print(colored(f"Nulls in measurement 17 before imputation = {df['measurement_17'].isna().sum():,.0f}",
                     color= 'blue'));
        df['measurement_17'] = df['measurement_17'].fillna(df['measurement_17_']);
        print(colored(f"Nulls in measurement 17 after imputation = {df['measurement_17'].isna().sum():,.0f}",
                     color = 'blue'));
        df = df.drop(['measurement_17_'], axis=1);
        
        # Adding columns for nulls in measurement 3 and 5 as seen in the reference notebook:-
        df['null_measurement_3'] = df['measurement_3'].isna() *1;
        df['null_measurement_5'] = df['measurement_5'].isna() *1;
        df['null_measurement_35'] = df['null_measurement_3'] * df['null_measurement_5'];
        
        df[['null_measurement_3','null_measurement_5','null_measurement_35']] = \
        df[['null_measurement_3','null_measurement_5','null_measurement_35']].astype(np.int8);
        
        #  Dropping measurement 3-9 after usage:-      
        df = df.drop(self.msmt_cols[3:10], axis=1);        
        self.cols = df.columns; 
        collect();
        print();
        return df;
    
    def get_feature_names_out(self, X, y=None): return self.cols;  
    
    def get_feature_names_in(self, X, y=None): return X.columns;   

In [ ]:
class DataProcessor(BaseEstimator, TransformerMixin):
    """
    This class aims to do the below tasks- 
    1. Null imputation of loading column using product level mean/ median
    2. Feature creation using nulls and attributes 2-3
    """;
    
    def __init__(self, strategy:str= 'mean', ndegree:np.int8=2): 
        self.strategy = strategy;
        self.ndegree = ndegree;
        
    def fit(self, X, y=None, **fit_params): 
        self.msmt_cols = X.columns[X.columns.str.startswith('measurement_')];
        return self;
    
    def transform(self, X, y= None, **transform_params):
        """
        This function aims to impute nulls in loading and creates new features for the relevant data-sets
        """;     
        df = X.copy(); 
        
        # Imputing nulls across loading column:-        
        _ = \
        df[['product_code','loading']].\
        merge(df[['product_code', 'loading']].dropna().groupby('product_code').mean(),
              how= 'left', left_on= 'product_code', right_index= True, suffixes= ('','_'));
        _['loading'] = _['loading'].fillna(_.loading_);
        df['loading'] = _.loading;
        del _;

        # Creating new features based on nulls:-        
        df['null_measurement'] = df[self.msmt_cols].isna().sum(axis=1);
        df['null_measurement'] = df['null_measurement'].astype(np.int8);
        
        _ = df[self.msmt_cols].isna().sum(axis=0);       
        for col in _.loc[_ > 0].index:
            df[f"null_{col}"] = df[col].isna()*1.0;
            df[f"null_{col}"] = df[f"null_{col}"].astype(np.int8);        
        del _;
        
        #  Creating new features for attributes 2 and 3:-
        df['attribute_23'] = df['attribute_2'] * df['attribute_3'];
        df['attribute_2Poly3'] = (df['attribute_2']**self.ndegree) * df['attribute_3'];
        df['attribute_23Poly'] = df['attribute_2'] * (df['attribute_3']**self.ndegree);       
        df = df.drop(['attribute_2', 'attribute_3'], axis=1, errors= 'ignore');
        
        df[['attribute_23','attribute_2Poly3','attribute_23Poly']] = \
        df[['attribute_23','attribute_2Poly3','attribute_23Poly']].astype(np.int32);       
        collect();
        self.cols = df.columns; 
        print();
        return df;
    
    def get_feature_names_out(self, X, y=None): return self.cols;  
    
    def get_feature_names_in(self, X:pd.DataFrame, y=None): return X.columns;

In [ ]:
class LGBMNullImputer(BaseEstimator, TransformerMixin):
    """
    This class imputes the nulls across the table using LGBM imputer
    """;
    
    def __init__(self, niter:np.int16): 
        self.niter = niter;
    
    def fit(self, X, y= None, **fit_params): return self;
    
    def transform(self, X1, y= None, **transform_params):     
        X = X1.copy();
        
        # Creating measurement columns with nulls and output structure:-
        _ = X.isna().sum(axis=0)
        cols = _.loc[_ > 0].index;
        del _;
        df = pd.DataFrame(columns = cols);

        # Creating the imputer instance:-
        imputer = LGBMImputer(n_iter = self.niter);

        for prod in tqdm(X.product_code.unique()):
            print(colored(f"Current product = {prod}", color= 'blue'));
            df = pd.concat([df, imputer.fit_transform(X.loc[X.product_code == prod, cols])], axis=0, ignore_index= False);

        # Modifying the output dataframe with the other columns and returning the result:-
        df = pd.concat([X.loc[:, X.columns[~X.columns.isin(cols)]], df], axis= 1);

        self.col_nm = df.columns;
        print();
        collect();
        return df; 
    
    def get_feature_names_out(self, X, y=None): return self.col_nm;  
    
    def get_feature_names_in(self, X:pd.DataFrame, y=None): return X.columns;         

In [ ]:
def ProcessData(df:pd.DataFrame):
    """
    This function encodes the attributes 0 and 1 and creates new features for measurements 10-15
    These combination features are created without any theoretical background, so they may be interpreted with caution
    """;
    
    df['attribute_0'] = LabelEncoder().fit_transform(df['attribute_0'].values);
    df['attribute_1'] = LabelEncoder().fit_transform(df['attribute_1'].values);  
    
    #  Creating linear and polynomial features from measurements 10-15:-
    _ = ['measurement_10', 'measurement_11', 'measurement_12', 'measurement_13','measurement_14', 'measurement_15'];
    df['measurement_lin'] = np.sum(df[_], axis=1).values.flatten();
        
    _cumsum = 0;
    for i,col in enumerate(_): _cumsum = _cumsum + (df[col]**2).values
    df['measurement_sq'] = _cumsum;
    del _cumsum;
    
    # Creating additional combination features from measurements 10-17:-
    df['msmt_1617Sq'] = df['measurement_16']*(df['measurement_17']**2);
    
#     df['ldSq_msmt17Sq'] = (df['loading']**2) / (df['measurement_17']**2);
#     df['ldSq_msmt1617'] = (df['loading']**2) / (df['measurement_17']*df['measurement_16']);
    df['ld_msmt17Sqrt'] = df['loading'] / (df['measurement_17']**0.50);
    df['ld_msmtSqrt1617'] = df['loading'] / ((df['measurement_17']**0.50)*(df['measurement_16']**0.50));
    df['ld_msmt16Pl17'] = df['loading'] / (df['measurement_16'] + df['measurement_17']);
    
    df['msmt_1016_17Sqrt'] = \
    (df['measurement_10']*df['measurement_11']*df['measurement_12']*\
     df['measurement_13']*df['measurement_14']*df['measurement_15']*df['measurement_16'])/(df['measurement_17']**2);
    
#     df['ld_msmtPl1017'] = \
#     df['loading'] / (df['measurement_10']+df['measurement_11']+df['measurement_12']+\
#     df['measurement_13']+df['measurement_14']+df['measurement_15']+df['measurement_16']+df['measurement_17']);
    
    # Creating logarithmic features:-
    df['lnmsmt1016'] = \
    np.log1p(df['measurement_10']+df['measurement_11']+df['measurement_12']+ df['measurement_13']+\
             df['measurement_14']+df['measurement_15']+df['measurement_16']);
            
    collect();
    return df;

In [ ]:
class DataScaler(BaseEstimator, TransformerMixin):
    "This class performs robust scaling on the datasets as per the user's choice of product code/ full data";
    
    def __init__(self, strategy:str= 'prod_cd'):
        self.strategy = strategy;
        
    def fit(self, X, y=None, **fit_params):
        return self;
        
    def transform(self, X, y= None, **transform_params):
        """
        This function aims to do the below-
        1. Implement a verbose version of robust scaler on measurement columns by product code/ full data
        2. Implement robustscaler completely on attribute2, attribute 3 derived columns
        """;
        
        df = X.copy();
        
        if self.strategy.lower() == 'prod_cd':
            cols = list(df.columns[df.columns.str.contains(r'^measurement_|msmt|product_code')]);
            _ = df[cols];
            _=\
            (_.set_index('product_code') - _.groupby('product_code').median()) / \
            (_.groupby('product_code').quantile([0.75,0.25]).groupby('product_code').agg(np.subtract.reduce));

            df = pd.concat([_.reset_index(),df[df.columns[~df.columns.isin(cols)]].reset_index()], axis=1).\
            drop(['index'], axis=1, errors= 'ignore');    
            del cols, _;
        
        elif self.strategy.lower() == 'full':
            cols = list(df.columns[df.columns.str.contains(r'^measurement_|msmt')]);
            _ = df[cols];
            _= (_ -_.median()) / iqr(_);

            df = pd.concat([_,df[df.columns[~df.columns.isin(cols)]]], axis=1);    
            del cols, _;

        # Implementing robustscaler completely on attributes 2,3 derived columns:-    
        cols = ['attribute_23', 'attribute_2Poly3', 'attribute_23Poly'];
        for col in cols: df[col] = (df[col] - df[col].median())/iqr(df[col]);
        
        self.col_nm = df.columns;
        collect(); 
        return df; 
    
    def get_feature_names_out(self, X, y=None): return self.col_nm;  
    
    def get_feature_names_in(self, X:pd.DataFrame, y=None): return X.columns;        

In [ ]:
class UnvSnpCreator(BaseEstimator, TransformerMixin):
    "This class calculates the univariate performance of features after feature processing";
    
    def __init__(self, strategy:str):
        self.strategy = strategy;
    
    def fit(self, X, y= None, **fit_params):
        global grid_specs;
        
        if self.strategy.lower() == 'full':
            print(colored(f"\nCorrelation plots\n",color= 'red', attrs= ['bold', 'dark']));        
            for method in tqdm(['pearson']):
                _ = X.corr(method= method);
                fig, ax= plt.subplots(1,1, figsize= (25,20));
                sns.heatmap(_, cbar=False, cmap= 'Blues', annot= True, fmt= '.0%',annot_kws= {'fontsize': 11},
                            linewidths=1.5,linecolor='white', mask= np.triu(np.ones_like(_)), ax=ax);
                ax.set_title(f'\n{method.capitalize()} Correlation plot across features in training data\n', 
                             color= 'tab:blue', fontweight = 'bold',fontsize = 14);
                plt.tight_layout();
                plt.show();
                del _;
                collect();

            print(colored(f"\nTarget-feature plots\n",color= 'red', attrs= ['bold', 'dark']));  
            fig, ax= plt.subplots(2,1, figsize= (16,16));
            pd.concat([X, y], axis=1).corr()[['failure']].drop(['failure']).plot.bar(ax=ax[0], color= 'tab:blue');
            ax[0].set_title(f"\nTarget correlation plots\n", color= 'tab:blue', fontweight = 'bold',fontsize = 14);
            ax[0].grid(**grid_specs);

            pd.DataFrame(data= mutual_info_classif(X.select_dtypes(np.number), y),
                         index= X.select_dtypes(np.number).columns,
                         columns= ['Mutual_Info']).plot.bar(ax=ax[1]);
            ax[1].set_title(f"\nMutual Information Plot\n", color= 'tab:blue', fontweight = 'bold',fontsize = 14);
            ax[1].grid(**grid_specs);

            plt.tight_layout();
            plt.show();
        return self;
    
    def transform(self, X, y= None, **transform_params):
        self.columns = X.columns;
        return X;
    
    def get_feature_names_out(self, X, y=None): return self.columns;  
    
    def get_feature_names_in(self, X:pd.DataFrame, y=None): return X.columns;  


## Pipeline implementation

This sub-section implements the pipeline and processes the data to elicit the model train-test sets
This is divided into 2 sequential steps-
1. Common component- This engenders data processing and feature creation
2. Strategy specifics- This creates a strategy specific scaling and univariate snapshot creation (full/ product)

4 datasets are created, 2 for full training data and 2 for classing at product code. They are then used subsequently.

In [ ]:
# Implementing the data processor pipeline:-
processor=\
Pipeline(verbose= True, steps= 
         [('M17Imputer',Msmt17Imputer()), 
          ('Processor1', DataProcessor('mean',2)),
          ('LGBMImputer', LGBMNullImputer(niter = 1000)),
          ('WOEGenerator', WoEEncoder(variables = ['attribute_0', 'attribute_1'], ignore_format= True)),
          ('Processor2', FunctionTransformer(ProcessData))
         ]);

print(colored(f"\nTrain-set pipeline implementation\n", color= 'red', attrs= ['bold', 'dark']));
Xtr = processor.fit_transform(xtrain, ytrain);
Xtr.index = xtrain.index;

print(colored(f"\nTest-set pipeline implementation\n", color= 'red', attrs= ['bold', 'dark']));
Xt = processor.transform(xtest);
Xt.index = xtest.index;

# Assessing nulls across the datasets after feature processing:-
print(colored(f"\nAssessing nulls across the datasets after feature processing\n", 
              color= 'red', attrs= ['bold', 'dark']));
display(pd.concat([Xtr.isna().sum(), Xt.isna().sum()], axis=1).\
        rename({0:'Train_Null', 1:'Test_Nulls'}, axis=1)); 

In [ ]:
# Studying measurement 17 and loading characteristics for failed and passed cases after data processing:-
_=\
pd.concat([pd.concat([Xtr[['loading']], ytrain], axis=1).query("failure == 1")[['loading']].\
           describe(percentiles=np.arange(0.05, 1.0,0.05)),
           pd.concat([Xtr[['loading']], ytrain], axis=1).query("failure == 0")[['loading']].\
           describe(percentiles=np.arange(0.05, 1.0,0.05)),
           pd.concat([Xtr[['measurement_17']], ytrain], axis=1).query("failure == 1")[['measurement_17']].\
           describe(percentiles=np.arange(0.05, 1.0,0.05)),
           pd.concat([Xtr[['measurement_17']], ytrain], axis=1).query("failure == 0")[['measurement_17']].\
           describe(percentiles=np.arange(0.05, 1.0,0.05))        
          ], axis=1);
_.columns = ['Loading_Fail', 'Loading_Pass', 'Measurement17_Fail', 'Measurement17_Pass'];

print(colored(f"\nLoading and measurement17 distribution study for failures/ passed cases\n",
              color= 'blue', attrs= ['bold', 'dark'])); 
display(_.style.format(precision=2, formatter = '{:,.2f}'));

del _;

In [ ]:
# Implementing scaling and univariate snapshot on the entire data:-
processor = Pipeline(verbose= True, 
                     steps= [('Scaler', DataScaler(strategy='full')),('UnvSnpCreator', UnvSnpCreator('full'))]);
print(colored(f"\nScaling and univariate performance on entire training data\n",color= 'blue', attrs= ['bold', 'dark'])); 
Xtrain = processor.fit_transform(Xtr,ytrain);
Xtest = processor.transform(Xt);

# Implementing scaling and univariate snapshot on the data classed by product code:-
processor = Pipeline(verbose= True, 
                     steps= [('Scaler', DataScaler(strategy='prod_cd')),('UnvSnpCreator', UnvSnpCreator('prod_cds'))]);
print(colored(f"\nScaling and univariate performance on data classed by product\n",color= 'blue', attrs= ['bold', 'dark'])); 
XtrainProd = processor.fit_transform(Xtr,ytrain);
XtestProd = processor.transform(Xt);

del processor, Xtr,Xt;
for g in range(3): collect(g);

# Saving the data for next notebook:-
Xtrain.to_csv('Xtrain.csv');
Xtest.to_csv('Xtest.csv');
XtrainProd.to_csv('XtrainProd.csv');
XtestProd.to_csv('XtestProd.csv');

# Step2- Feature dependency

* We delve into the features classed by product codes and construct the correlation plots thereby. 
* We shall use a selected list of columns at product code level and consider a model development. Attribute columns are excluded at product level as they are constant for a given product.

In [ ]:
# Shortlisting the measurement based columns for the subsequent steps:-
Mdl_Ftre_Lst = XtrainProd.columns[(XtrainProd.columns.str.contains(r'measurement|loading|product_code|msmt'))];
print(colored(f"\nShortlisting the measurement based columns for the subsequent steps\n",
              color= 'blue', attrs= ['bold', 'dark']));
display(Mdl_Ftre_Lst);

# Concatenating train-test data with the truncated columns:-
train_test = pd.concat([XtrainProd[Mdl_Ftre_Lst], XtestProd[Mdl_Ftre_Lst]], axis=0);

In [ ]:
# Plotting correlations at product level:-
fig, ax= plt.subplots(3,3, figsize= (45,45), sharex= True, sharey=True);
for i, prod in tqdm(enumerate(train_test.product_code.unique())):
    _ = train_test.corr();    
    sns.heatmap(_, cbar=False, cmap= 'Blues', annot= True, fmt= '.0%',annot_kws= {'fontsize': 9, 'fontweight': 'bold'},
                linewidths= 1.75,linecolor='white', mask= np.triu(np.ones_like(_)), 
                ax=ax[i//3, i%3]);
    ax[i//3, i%3].set_title(f'\nCorrelation plot for product {prod}\n',color= 'tab:blue', fontweight = 'bold',fontsize = 14);
    del _;
plt.tight_layout();
plt.show();

del train_test;
collect();

In [ ]:
# Plotting target-feature correlations at product-code classing:-
print(colored(f"\nTarget-feature plots\n",color= 'red', attrs= ['bold', 'dark']));  

for i, prod in tqdm(enumerate(XtrainProd.product_code.unique())):   
    X = XtrainProd.loc[XtrainProd.product_code == prod, Mdl_Ftre_Lst];
    y = ytrain.loc[X.index];
    
    fig, ax= plt.subplots(1,2, figsize= (22,6.5), sharex= True);  
    pd.concat([X, y], axis=1).corr()[['failure']].drop(['failure']).plot.bar(ax=ax[0], color= 'tab:blue');
    ax[0].set_title(f"\nTarget correlation plots- product {prod}\n", color= '#030303', fontweight = 'bold',fontsize = 12);
    ax[0].grid(**grid_specs);

    pd.DataFrame(data= mutual_info_classif(X.select_dtypes(np.number), y),
                 index= X.select_dtypes(np.number).columns,
                 columns= ['Mutual_Info']).plot.bar(ax=ax[1], color = '#00B2EE');
    ax[1].set_title(f"\nMutual Information Plot- product {prod}\n", color= '#009ACD', fontweight = 'bold',fontsize = 12);
    ax[1].grid(**grid_specs);
    del X,y;
    collect();
    
    plt.tight_layout();
    plt.show();

## Key notes:- 
1. No major feature-target correlations are discovered as yet. Combination features are performing better than some individual ones, but are not good enough
2. Loading and measurement 17 seem to be important features. Combinations from them are also fine, but creating collinearity problem
3. Additional features are partly useful, this will be clear during the model development phase
4. Ensemble needs to be done with care as the public leaderboard may not reflect the true test set picture

# Step3- Next steps

1. Model development with the features- We will explore logistic regression as others have done it too
2. We will try and generate a CV score to perhaps hill climb the private leaderboard